# Antimicrobial-resistance genes in bacterial isolates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/microbiology/resistance_genes.ipynb)

**The problem.** You sequenced and assembled bacterial isolates (from patients, a hospital ward, a farm, a
surveillance program). Which acquired resistance genes does each carry, and against which drug classes?
Is this *Klebsiella* a carbapenemase producer (KPC, NDM, OXA-48)? Is this *S. aureus* methicillin-resistant
(mecA, mecC)? This notebook searches each isolate for the genes of the NCBI AMRFinderPlus catalog.

**What you need.** An assembly (FASTA of contigs) per isolate, or its reads (FASTQ); the AMRFinderPlus
catalog (`AMR_CDS.fa`, and `ReferenceGeneCatalog.txt` for the drug classes).

**What you get.** The genes found in each isolate, with identity, coverage and position; the genes grouped by
drug class; a gene-by-isolate table; and the answers to the first questions (carbapenemase? mecA? ESBL?).

*Experimental, for teaching and research: a gene found does not prove resistance, and resistance can come
from mutations this search does not check. It does not replace susceptibility testing.*

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from the GitHub
# release (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.1.0"
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Download the wheel for your system from\n"
              "  https://github.com/fbcouto/multimerge-genomics-research/releases/tag/v2.1.0\n"
              "install it with:  python -m pip install <the .whl file>\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{RELEASE}/multimerge_genomics-2.1.0-cp310-abi3-manylinux_2_28_x86_64.whl"], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the research demo pack (downloaded once, about 45 MB): four public genome assemblies (NCBI RefSeq):
*K. pneumoniae* KPNIH1 and ATCC BAA-2146, *S. aureus* USA300 and NCTC 8325; and the AMRFinderPlus
reference gene catalog, release 2026-10-29.1 (NCBI, public domain). To use your own isolates, set
`MY_ISOLATES`; download the current catalog from NCBI (`AMR_CDS.fa`, `ReferenceGeneCatalog.txt`).

In [ ]:
# Your isolates (None = the demo data) and the search thresholds
MY_ISOLATES = None           # {"isolate name": "assembly.fasta"} or {"name": ["R1.fastq.gz", "R2.fastq.gz"]}
MY_CATALOG = None            # AMR_CDS.fa of AMRFinderPlus (plain or .gz)
MY_CATALOG_TABLE = None      # its ReferenceGeneCatalog.txt (drug class of each gene)
CATALOG_VERSION = "2026-10-29.1"   # write the version of the catalog you use
MIN_IDENTITY = 0.90          # minimum identity to the catalog allele
MIN_COVERAGE = 0.90          # minimum fraction of the gene covered
INCLUDE_PLUS = True          # also the "plus" genes: virulence, stress, metals, biocides

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "resistance_genes"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_ISOLATES is None
ISOLATES = MY_ISOLATES or {
    "K. pneumoniae KPNIH1": demo_file("mmgen_demo_research_v4.zip", "research/microbio/isolates/kpneumoniae_KPNIH1_KPC-3.fna.gz"),
    "K. pneumoniae ATCC BAA-2146": demo_file("mmgen_demo_research_v4.zip", "research/microbio/isolates/kpneumoniae_ATCC_BAA-2146_NDM-1.fna.gz"),
    "S. aureus USA300": demo_file("mmgen_demo_research_v4.zip", "research/microbio/isolates/saureus_USA300_MRSA.fna.gz"),
    "S. aureus NCTC 8325": demo_file("mmgen_demo_research_v4.zip", "research/microbio/isolates/saureus_NCTC8325_MSSA.fna.gz"),
}
CATALOG = MY_CATALOG or demo_file("mmgen_demo_research_v4.zip", "research/microbio/AMR_CDS.fa.gz")
CATALOG_TABLE = MY_CATALOG_TABLE or demo_file("mmgen_demo_research_v4.zip", "research/microbio/ReferenceGeneCatalog.txt.gz")
for name, files in ISOLATES.items():
    print(f"{name}: {files}")

## Step 1. Search each isolate

Each catalog gene is searched in the isolate's contigs (both strands); for each locus the best-matching
allele is reported with its identity and the fraction of the gene covered. A hit below `MIN_COVERAGE`
but above half the gene is reported as partial (a gene broken across two contigs, or truncated). From
reads, the genes are covered by the reads instead, with a depth.

In [ ]:
import time

results = {}
for name, files in ISOLATES.items():
    start = time.perf_counter()
    r = mmg.find_resistance_genes([files] if isinstance(files, str) else list(files), CATALOG,
                                  catalog_table=CATALOG_TABLE, min_identity=MIN_IDENTITY,
                                  min_coverage=MIN_COVERAGE, include_plus=INCLUDE_PLUS)
    results[name] = r
    core = [h for h in r["hits"] if h["scope"] == "core"]
    print(f"{name}: {r['sequences']} sequence(s), {r['bases']:,} bases ({r['mode']}); "
          f"{len(core)} resistance gene(s) and {len(r['hits']) - len(core)} 'plus' gene(s) "
          f"({time.perf_counter() - start:.1f} s)")

with open(OUT / "resistance_genes.tsv", "w") as f:
    columns = ["isolate", "gene", "scope", "class", "subclass", "identity", "coverage", "contig", "start", "end", "strand", "status", "product"]
    f.write("\t".join(columns) + "\n")
    for name, r in results.items():
        for h in r["hits"]:
            f.write("\t".join(str(name if c == "isolate" else h[c]) for c in columns) + "\n")
print("Every hit written to", OUT / "resistance_genes.tsv")

## Step 2. By drug class, and gene by isolate

Only the "core" genes (acquired resistance); the "plus" genes are in the table file.

The identity (%) of each resistance gene in each isolate: the same gene across isolates of an outbreak
points to a shared strain or a shared plasmid.

In [ ]:
for name, r in results.items():
    classes = {}
    for h in r["hits"]:
        if h["scope"] == "core":
            classes.setdefault(h["class"] or "OTHER", set()).add(h["gene"] + ("" if h["status"] in ("exact", "aligned") else f" ({h['status']})"))
    print(f"\n{name}")
    for drug_class, found in sorted(classes.items()):
        print(f"   {drug_class:<45} {', '.join(sorted(found))}")
    if not classes:
        print("   no acquired resistance gene found")

table = {}
for name, r in results.items():
    for h in r["hits"]:
        if h["scope"] == "core":
            row = table.setdefault(h["gene"], {"gene": h["gene"], "class": h["class"], **dict.fromkeys(results, "-")})
            row[name] = round(100 * h["identity"], 2)
show(sorted(table.values(), key=lambda row: (row["class"] or "", row["gene"])), n=100)

## Step 3. The first questions

Carbapenemases (last-line beta-lactam resistance), methicillin resistance in staphylococci (mecA, mecC),
extended-spectrum cephalosporin resistance (ESBL and AmpC enzymes such as CTX-M or CMY) and colistin
resistance (mcr), read from the catalog's drug subclasses.

In [ ]:
questions = {"carbapenemase": ("subclass", "CARBAPENEM"), "methicillin (mecA/mecC)": ("subclass", "METHICILLIN"),
             "extended-spectrum cephalosporins": ("subclass", "CEPHALOSPORIN"), "colistin (mcr)": ("class", "COLISTIN")}
answers = []
for name, r in results.items():
    row = {"isolate": name}
    for question, (field, value) in questions.items():
        genes = sorted({h["gene"] for h in r["hits"] if h["scope"] == "core" and value in (h[field] or "")})
        row[question] = ", ".join(genes) or "-"
    answers.append(row)
show(answers)

## Check the result

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


check(all(r["sequences"] > 0 and r["bases"] > 0 for r in results.values()), "every isolate had sequence to search")
check(all(r["catalog_unmatched"] == 0 for r in results.values()), "every catalog gene has its row in the catalog table")
check(all(h["identity"] >= MIN_IDENTITY for r in results.values() for h in r["hits"]), f"every hit reaches {MIN_IDENTITY:.0%} identity")
partial = [(n, h["gene"]) for n, r in results.items() for h in r["hits"] if h["coverage"] < MIN_COVERAGE]
check(not partial, f"no partial gene ({partial[:5]}): a partial gene may be split across contigs or truncated")
if USING_DEMO:
    genes = {name: {h["gene"] for h in r["hits"]} for name, r in results.items()}
    assert any(g.startswith("blaKPC") for g in genes["K. pneumoniae KPNIH1"])
    assert "blaNDM-1" in genes["K. pneumoniae ATCC BAA-2146"]
    assert "mecA" in genes["S. aureus USA300"] and "mecA" not in genes["S. aureus NCTC 8325"]
    print("Demo expectations met: KPC in KPNIH1, NDM-1 in BAA-2146, mecA only in USA300.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""{len(results)} isolates were searched for the nucleotide sequences of the NCBI AMRFinderPlus reference
gene catalog (release {CATALOG_VERSION}) with find_resistance_genes of MMGEN version {VERSION}
(Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492), reporting the best allele per locus with at least {MIN_IDENTITY:.0%} identity and
{MIN_COVERAGE:.0%} coverage; drug classes were taken from the catalog's ReferenceGeneCatalog table. Point mutations
associated with resistance were not assessed, and the predictions were not confirmed by susceptibility testing."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Reads instead of assemblies: `MY_ISOLATES = {"isolate 1": ["R1.fastq.gz", "R2.fastq.gz"]}`; the genes are
  then covered by reads and get a depth.
* Use the current catalog: download `AMR_CDS.fa` and `ReferenceGeneCatalog.txt` from the NCBI AMRFinderPlus
  data directory and write its version in `CATALOG_VERSION`.
* Novel alleles: lower `MIN_IDENTITY` (e.g. 0.8) to see distant relatives of known genes; report them as such.
* Resistance by mutation (gyrA, parC, ompK35/36, rpoB...) needs AMRFinderPlus with `--organism`, or a
  targeted variant check.
* To relate isolates of an outbreak, add their distances and tree (see `outbreaks/`).

## References

* Feldgarden M. et al. (2021) AMRFinderPlus and the Reference Gene Catalog facilitate examination of the
  genomic links among antimicrobial resistance, stress response, and virulence. *Scientific Reports* 11:12728.
* Feldgarden M. et al. (2019) Validating the AMRFinder tool and resistance gene database by using
  antimicrobial resistance genotype-phenotype correlations in a collection of isolates.
  *Antimicrobial Agents and Chemotherapy* 63:e00483-19.
* Snitkin E.S. et al. (2012) Tracking a hospital outbreak of carbapenem-resistant *Klebsiella pneumoniae* with
  whole-genome sequencing. *Science Translational Medicine* 4:148ra116 (KPNIH1).